# Solución — Ejercicio 02: Model Registry

Solución de referencia de
[`../exercises/ejercicio-02.md`](../exercises/ejercicio-02.md).

**No la publiques antes del taller.**

Prerrequisito: `make mlflow` (el registry necesita el backend SQLite; con un file
store no existe).

In [ ]:
import mlflow
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from taxi import config

NOMBRE_MODELO = "iris-clasificacion"

## Parte 1 — Datos y modelo

Un detalle que el enunciado deja pasar y conviene señalar en clase: el
`StandardScaler` va **dentro de un `Pipeline`**, no aplicado por fuera. Si el
scaler se ajusta aparte y solo se registra el clasificador, el artefacto del
registry **no sabe escalar** y hay que reimplementar el preprocesamiento en cada
consumidor. Un artefacto, una versión, un hash.

In [ ]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=config.SEMILLA, stratify=y
)

n_estimators = 100
max_depth = 5
random_state = config.SEMILLA

modelo = Pipeline(
    [
        ("escalador", StandardScaler()),
        (
            "clasificador",
            RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=random_state,
            ),
        ),
    ]
)
modelo.fit(X_train, y_train)
accuracy = float(accuracy_score(y_test, modelo.predict(X_test)))
print(f"accuracy v1: {accuracy:.4f}")
print("distribucion de clases:", np.bincount(y))

## TODO 1 — Conexión

In [ ]:
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
mlflow.set_experiment("iris-model-registry")
cliente = mlflow.MlflowClient()
print("tracking URI:", mlflow.get_tracking_uri())

## TODO 2 — Loguear el modelo dentro de un run

`name="modelo_rf"`, **no** `artifact_path=` ni el segundo argumento posicional:
`artifact_path` está deprecado en los flavors de MLflow 3.

Se agregan `signature` e `input_example` aunque el enunciado no los pida: un
modelo que va a ir al registry sin firma es un modelo que no se puede servir con
garantías.

In [ ]:
ejemplo = X_test.head(5).astype("float64")
firma = infer_signature(ejemplo, modelo.predict(ejemplo))

with mlflow.start_run(run_name="rf_iris_v1") as run:
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(
        sk_model=modelo,
        name="modelo_rf",
        signature=firma,
        input_example=ejemplo,
    )
    run_id = run.info.run_id

print("run_id:", run_id)

> Aquí `log_model` no necesita `skops_trusted_types` porque todos los tipos del
> pipeline son de scikit-learn. En el caso guía sí hace falta, porque el pipeline
> incluye una clase propia (`ADiccionarios`) y, si el estimador es XGBoost, dos
> tipos más.

## TODO 3 — Registrar

`registrar` crea el nombre y la versión. El run sigue siendo el mismo: la versión
**apunta** a él, y eso es lo que da la trazabilidad.

In [ ]:
model_uri = f"runs:/{run_id}/modelo_rf"
version = mlflow.register_model(model_uri, NOMBRE_MODELO)
print(f"{NOMBRE_MODELO} v{version.version} registrada desde {model_uri}")

## TODO 4 — Tag primero, alias después

El orden es la parte importante. Si el proceso muere entre las dos llamadas:

- tag → alias: "validada pero no promovida". Seguro.
- alias → tag: un modelo sirviendo tráfico sin registro de haber sido validado.

In [ ]:
cliente.set_model_version_tag(NOMBRE_MODELO, version.version, "validation_status", "passed")
cliente.set_registered_model_alias(NOMBRE_MODELO, "champion", version.version)

champion = cliente.get_model_version_by_alias(NOMBRE_MODELO, "champion")
print("aliases:", cliente.get_registered_model(NOMBRE_MODELO).aliases)
print(f"champion = v{champion.version} | tags: {champion.tags}")

## TODO 5 y 6 — Cargar por versión y por alias

La diferencia no es estética:

- `models:/nombre/1` fija la versión. Sirve para auditar o reproducir un
  incidente.
- `models:/nombre@champion` **no menciona ninguna versión**. Es la que va en
  producción: cuando promuevas la v2, este código no cambia.

Y se carga como `pyfunc`, no con `mlflow.sklearn.load_model`: el consumidor no
tiene por qué saber con qué librería se entrenó.

In [ ]:
uri_por_version = f"models:/{NOMBRE_MODELO}/1"
modelo_v1 = mlflow.pyfunc.load_model(uri_por_version)
print("cargado por version:", uri_por_version)

uri_por_alias = f"models:/{NOMBRE_MODELO}@champion"
modelo_champion = mlflow.pyfunc.load_model(uri_por_alias)
print("cargado por alias  :", uri_por_alias)

## TODO 7 — Predecir y comparar

Si la accuracy no es idéntica, el modelo del registry no es el que entrenaste.

In [ ]:
y_pred_registry = modelo_champion.predict(X_test)
accuracy_registry = float(accuracy_score(y_test, y_pred_registry))

print(f"accuracy original : {accuracy:.6f}")
print(f"accuracy registry : {accuracy_registry:.6f}")
print("identicas:", accuracy == accuracy_registry)

## TODO 8 — La versión 2, logueada y registrada en un paso

In [ ]:
n_estimators_v2 = 200
max_depth_v2 = 10

with mlflow.start_run(run_name="rf_iris_v2") as run2:
    mlflow.log_param("n_estimators", n_estimators_v2)
    mlflow.log_param("max_depth", max_depth_v2)
    mlflow.log_param("random_state", random_state)

    modelo_v2 = Pipeline(
        [
            ("escalador", StandardScaler()),
            (
                "clasificador",
                RandomForestClassifier(
                    n_estimators=n_estimators_v2,
                    max_depth=max_depth_v2,
                    random_state=random_state,
                ),
            ),
        ]
    )
    modelo_v2.fit(X_train, y_train)
    accuracy_v2 = float(accuracy_score(y_test, modelo_v2.predict(X_test)))
    mlflow.log_metric("accuracy", accuracy_v2)

    mlflow.sklearn.log_model(
        sk_model=modelo_v2,
        name="modelo_rf",
        signature=firma,
        input_example=ejemplo,
        registered_model_name=NOMBRE_MODELO,
    )
    run_id_v2 = run2.info.run_id

print(f"run v2: {run_id_v2} | accuracy v2: {accuracy_v2:.4f}")

## TODO 9 — La v2 queda como `candidate`, no como `champion`

Da igual si su accuracy salió mejor o peor: es el punto conceptual del ejercicio.
**Registrar no es promover.**

Y fíjate en el número que obtuviste: con 30 filas de test, una diferencia de un
par de puntos de accuracy equivale a **una sola predicción**. Cabe entera dentro
del ruido de muestreo. Promover con esa evidencia es precisamente el error que el
gate de S06 existe para impedir — y ahí el juez es un holdout fijo, con una mejora
mínima exigida y un chequeo por subgrupos.

In [ ]:
versiones = cliente.search_model_versions(f"name='{NOMBRE_MODELO}'")
ultima = max(versiones, key=lambda mv: int(mv.version))

cliente.set_registered_model_alias(NOMBRE_MODELO, "candidate", ultima.version)
cliente.set_model_version_tag(NOMBRE_MODELO, ultima.version, "validation_status", "pending")

print("aliases:", cliente.get_registered_model(NOMBRE_MODELO).aliases)

## Verificación final

In [ ]:
info = cliente.get_registered_model(NOMBRE_MODELO)
print(f"modelo: {info.name}")
print(f"aliases: {info.aliases}\n")

for alias in ["champion", "candidate"]:
    mv = cliente.get_model_version_by_alias(NOMBRE_MODELO, alias)
    metricas = mlflow.get_run(mv.run_id).data.metrics
    estado = mv.tags.get("validation_status", "SIN TAG")
    print(f"  {alias:10s} -> v{mv.version} | accuracy: {metricas.get('accuracy')} | {estado}")

## Bonus 1 — Promoción y rollback

Las dos operaciones son escrituras de metadatos. Cronometrarlas en clase es más
convincente que explicarlo: el rollback de un modelo no requiere reentrenar, ni
reconstruir una imagen, ni redesplegar nada.

In [ ]:
cliente.set_registered_model_alias(NOMBRE_MODELO, "champion", ultima.version)
print("promovido:", cliente.get_registered_model(NOMBRE_MODELO).aliases)

cliente.set_registered_model_alias(NOMBRE_MODELO, "champion", "1")
print("rollback :", cliente.get_registered_model(NOMBRE_MODELO).aliases)

## Bonus 3 — Buscar por tag

El tag `validation_status` no es decorativo: es lo que permite responder "qué
versiones pasaron un gate" sin abrir la UI. Es también la evidencia que pide una
auditoría.

In [ ]:
aprobadas = [
    (mv.version, mv.tags.get("validation_status"))
    for mv in cliente.search_model_versions(f"name='{NOMBRE_MODELO}'")
    if mv.tags.get("validation_status") == "passed"
]
print("versiones con validation_status=passed:", aprobadas)

## Cierre para el instructor

| Síntoma | Causa |
|---|---|
| `RestException` al registrar | el server no tiene backend de base de datos |
| "el alias no aparece" | se asignó sobre otro nombre de modelo |
| "la accuracy del registry no coincide" | se cargó otra versión, o el scaler quedó fuera del pipeline |
| `MlflowException: untrusted types` | pasa en el caso guía, no aquí: default `skops` + clase propia |
| Alguien usa `transition_model_version_stage` | lo copió de un tutorial: es la tabla de "qué NO usar" |